# 🤝 Notebook 08: Ensemble Learning
## Customer Churn Prediction
- Voting Classifier (Hard & Soft)
- Stacking Classifier

In [ ]:
import numpy as np
import pandas as pd
import joblib
import os
from pathlib import Path
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import recall_score, roc_auc_score
print("✅ Libraries imported!")

if os.path.exists('/kaggle/working'):
    X_train = np.load('/kaggle/working/X_train.npy')
    X_test = np.load('/kaggle/working/X_test.npy')
    y_train = np.load('/kaggle/working/y_train.npy')
    y_test = np.load('/kaggle/working/y_test.npy')
    MODEL_DIR = Path('/kaggle/working/models')
    RESULTS_PATH = '/kaggle/working/models/model_results.csv'
else:
    X_train = np.load('../models/X_train.npy')
    X_test = np.load('../models/X_test.npy')
    y_train = np.load('../models/y_train.npy')
    y_test = np.load('../models/y_test.npy')
    MODEL_DIR = Path('../models')
    RESULTS_PATH = Path('../models/model_results.csv')

results_df = pd.read_csv(RESULTS_PATH)
print(f"✅ Loaded results: {df_results.shape}")

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
rf = RandomForestClassifier(random_state=42, n_jobs=-1)
xgb = XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=-1)
lgbm = LGBMClassifier(random_state=42, n_jobs=-1)
estimators = [('lr', lr), ('rf', rf), ('xgb', xgb), ('lgbm', lgbm)]

In [ ]:
print("🗳️ Training Voting Classifier (Soft)...")
voting_soft = VotingClassifier(estimators=estimators, voting='soft')
voting_soft.fit(X_train, y_train)
y_pred_vs = voting_soft.predict(X_test)
y_proba_vs = voting_soft.predict_proba(X_test)[:, 1]
res_vs = {'Model': 'Voting_Soft', 'Accuracy': voting_soft.score(X_test, y_test), 'Precision': 0, 'F1': 0, 'Recall': recall_score(y_test, y_pred_vs), 'ROC-AUC': roc_auc_score(y_test, y_proba_vs)}
print(f"   ✅ Recall: {res_vs['Recall']:.4f}")

print("🗳️ Training Voting Classifier (Hard)...")
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_hard.fit(X_train, y_train)
y_pred_vh = voting_hard.predict(X_test)
res_vh = {'Model': 'Voting_Hard', 'Accuracy': voting_hard.score(X_test, y_test), 'Precision': 0, 'F1': 0, 'ROC-AUC': 0, 'Recall': recall_score(y_test, y_pred_vh)}
print(f"   ✅ Recall: {res_vh['Recall']:.4f}")

In [ ]:
print("🥞 Training Stacking Classifier...")
stacking_clf = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), cv=5, stack_method='predict_proba', n_jobs=-1)
stacking_clf.fit(X_train, y_train)
y_pred_st = stacking_clf.predict(X_test)
y_proba_st = stacking_clf.predict_proba(X_test)[:, 1]
res_st = {'Model': 'Stacking', 'Accuracy': stacking_clf.score(X_test, y_test), 'Precision': 0, 'F1': 0, 'Recall': recall_score(y_test, y_pred_st), 'ROC-AUC': roc_auc_score(y_test, y_proba_st)}
print(f"   ✅ Recall: {res_st['Recall']:.4f}")

In [ ]:
new_rows = pd.DataFrame([res_vs, res_vh, res_st])
cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
final_results = pd.concat([results_df, new_rows[cols]], ignore_index=True)
final_results = final_results.sort_values('Recall', ascending=False)
final_results.to_csv(RESULTS_PATH, index=False)
best_name = final_results.iloc[0]['Model']
if best_name == 'Stacking': joblib.dump(stacking_clf, MODEL_DIR / 'best_model.pkl')
elif best_name == 'Voting_Soft': joblib.dump(voting_soft, MODEL_DIR / 'best_model.pkl')
else: joblib.dump(voting_hard, MODEL_DIR / 'best_model.pkl')
print(f"🏆 New Best Model: {best_name}")
print("✅ Notebook 08 Complete!")